# 5. Advanced RAG 기법 — Agent 자율 검색 판단 + Context Memory

Agent가 검색 전략을 자율적으로 결정하는 **Agentic RAG**와 대화 맥락을 유지하는 **Context Memory**를 학습합니다.

| 파트 | 주제 | 핵심 내용 |
|------|------|-----------|
| **Part 1** | Agent 자율 검색 판단 | Naive RAG 한계 → GradeDocuments → StateGraph 재검색 루프 |
| **Part 2** | Context Memory | Context Engineering 매트릭스 → 메모리 3유형 → InMemoryStore |


## 0) 환경 설정
> 진행: [▓░░░░░░░░░] 1/11

In [ ]:
# [5-1] : 라이브러리 설치

!uv pip install -q langchain langchain-core langchain-openai langchain-chroma chromadb langchain-text-splitters langgraph python-dotenv pydantic

In [ ]:
# [5-2] : 환경변수 로드 및 API 키 검증

import os
from dotenv import load_dotenv

load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
assert OPENAI_API_KEY, "OPENAI_API_KEY가 .env에 설정되어 있지 않습니다."

# 트레이싱 설정 (선택)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON — project: {os.environ['LANGCHAIN_PROJECT']}")

langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON — {os.environ.get('LANGFUSE_HOST', '')}")

lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}

print(f"OPENAI_API_KEY: {'OK' if OPENAI_API_KEY else 'MISSING'} (필수)")

In [ ]:
# [5-3] : 공통 LLM 및 임베딩 모델 초기화

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)
embedding = OpenAIEmbeddings(model="text-embedding-3-small")

print(f"LLM: {llm.model_name}")
print(f"Embedding: {embedding.model}")

---
# Part 1: Agent 자율 검색 판단

Naive RAG의 한계를 분석하고, Agent가 검색 전략을 자율적으로 결정하는 **Agentic RAG** 아키텍처를 구축한다.


## 1) Naive RAG 한계 분석

# [1-1] : Naive RAG 한계 분석

**Naive RAG**는 가장 기본적인 검색 증강 생성 파이프라인이다: `질문 → 검색(top-k) → LLM 생성`. 그러나 실무에서 다음과 같은 한계가 드러난다:

| 한계 | 설명 |
|------|------|
| **검색 정밀도 부족** | 임베딩 유사도만으로 top-k를 선택 — 의미적으로 관련 없는 문서가 포함됨 |
| **관련성 검증 부재** | 검색 결과의 품질을 평가하지 않고 그대로 LLM에 전달 |
| **재검색 불가** | 검색 결과가 부적절해도 한 번의 검색으로 끝남 — 자기 수정 능력 없음 |
| **고정 쿼리** | 사용자 질문을 그대로 검색 쿼리로 사용 — 검색에 최적화된 쿼리 변환 없음 |
| **환각(Hallucination)** | 관련 없는 문서 기반으로 그럴듯한 답변을 생성 |

### Naive RAG vs Agentic RAG

| 단계 | Naive RAG | Agentic RAG |
|------|-----------|-------------|
| **검색 결정** | 항상 검색 | Agent가 필요 여부를 자율 판단 |
| **쿼리 생성** | 원본 질문 그대로 | Agent가 검색에 최적화된 쿼리 생성 |
| **관련성 평가** | 없음 | GradeDocuments로 문서 품질 검증 |
| **재검색** | 불가 | 관련성 낮으면 쿼리 재작성 후 재검색 |
| **생성** | 전체 문서 전달 | 관련 문서만 선별 전달 |

> 진행: [▓▓░░░░░░░░] 2/11

## 2) Agentic RAG 아키텍처

# [1-2] : Agentic RAG 아키텍처

**Agentic RAG**는 Agent가 검색 전략을 자율적으로 결정하는 아키텍처이다. 핵심은 검색 결과를 **평가**하고, 부적절할 경우 **재검색**하는 자기 수정 루프이다.

### 핵심 컴포넌트

| 컴포넌트 | 역할 | 구현 방식 |
|---------|------|----------|
| **Retriever** | 벡터스토어에서 문서 검색 | `similarity_search(query, k=3)` |
| **GradeDocuments** | 검색 문서의 관련성 평가 | Pydantic + `with_structured_output` |
| **Query Rewriter** | 관련성 낮을 때 쿼리 재작성 | LLM 기반 쿼리 변환 |
| **Re-ranker** | 검색 결과 재정렬 | LLM 기반 관련성 점수 |
| **StateGraph** | 전체 흐름 제어 | 조건 분기 + 순환 엣지 |

### 아키텍처 흐름

```
START → gen_query → [tools_condition] → retrieve → grade_documents
                              ↓ __end__              ↓ relevant
                             END           generate_answer → END
                                                ↓ not_relevant
                                          rewrite_query → gen_query
```

> 진행: [▓▓▓░░░░░░░] 3/11

In [ ]:
# [5-4] : 벡터스토어 구축 — 샘플 문서 + ChromaDB
# [핵심] Agentic RAG의 검색 기반이 되는 벡터스토어를 구축
# [주의] persist_directory="./chroma_advanced_rag_db"로 로컬 파일 기반 DB를 생성하며, reset_directory=True로 재실행 시 중복을 방지

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from notebook_vectorstore import build_vectorstore_from_documents

# TODO : 벡터스토어 구축 — 샘플 문서 + ChromaDB (변수명: raw_docs, page_content, metadata, page_content, metadata)

print(f"청킹 결과: {len(raw_docs)}개 문서 → {len(splits)}개 청크")
print(f"벡터스토어 구축 완료: {CHROMA_DIR} | 컬렉션: {COLLECTION_NAME}")
from langchain_core.tools import tool

# TODO : 함수/클래스 정의: retrieve (변수명: retrieve)

print("retrieve 도구 등록 완료")


## 3) 문서 관련성 평가 (GradeDocuments)

# [1-3] : 문서 관련성 평가 (GradeDocuments)

**GradeDocuments**는 검색된 문서의 관련성을 LLM이 자동으로 평가하는 핵심 패턴이다. `with_structured_output`으로 구조화된 `yes/no` 판정을 받아 조건 분기에 사용한다.

| 판정 | 의미 | 다음 행동 |
|------|------|-----------|
| `yes` | 문서가 질문과 관련 있음 | → `generate_answer` (답변 생성) |
| `no` | 문서가 질문과 관련 없음 | → `rewrite_query` (쿼리 재작성 후 재검색) |

> 진행: [▓▓▓▓░░░░░░] 4/11

In [ ]:
# [5-5] : GradeDocuments — Pydantic 스키마 + LLM 관련성 평가

from pydantic import BaseModel, Field
from typing import Literal

# TODO : 함수/클래스 정의: GradeDocuments (변수명: GradeDocuments)

print(f"관련성 평가: score={test_result.score} | {test_result.reasoning}")
print(f"관련성 평가: score={test_result2.score} | {test_result2.reasoning}")

## 4) 쿼리 재작성 (Query Rewrite)

# [1-4] : 쿼리 재작성 (Query Rewrite)

검색된 문서가 관련 없을 때, 원래 질문을 더 구체적으로 **재작성**하여 검색 품질을 향상시킨다. 이것이 Agentic RAG의 핵심인 **자기 수정 루프**이다.

```
원본: "RAG의 성능을 높이는 방법은?"
  → 재작성: "RAG 시스템에서 검색 정밀도를 높이기 위한 Re-ranking과 Query Decomposition 기법"
```

> 진행: [▓▓▓▓▓░░░░░] 5/11

In [ ]:
# [5-6] : 쿼리 재작성 함수 정의

from langchain_core.messages import HumanMessage

# TODO : 함수/클래스 정의: rewrite_query (변수명: rewrite_query)

print(f"원본: {original_q}")
print(f"재작성: {rewritten_q}")

## 5) Agent 자율 판단 StateGraph

# [1-5] : Agent 자율 판단 StateGraph

LangGraph `StateGraph`로 Agentic RAG 파이프라인을 구현한다. 핵심은 `grade_documents` 노드의 결과에 따라 **generate**(답변 생성) 또는 **rewrite**(쿼리 재작성 후 재검색)로 분기하는 **조건부 엣지**이다.

| 노드 | 역할 |
|------|----- |
| `gen_query` | 도구 호출 또는 직접 응답 결정 |
| `retrieve` | 벡터스토어 검색 실행 (ToolNode) |
| `grade_documents` | 검색 결과 관련성 평가 (yes/no) |
| `generate_answer` | 관련 문서 기반 최종 답변 생성 |
| `rewrite_query` | 관련 없을 때 질문 재작성 후 재검색 루프 |

> 진행: [▓▓▓▓▓▓░░░░] 6/11

In [ ]:
# [5-7] : StateGraph 상태 정의 + 노드 함수 구현

from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage, AIMessage
from langgraph.graph.message import add_messages

# TODO : 함수/클래스 정의: RAGState, generate_query_or_respond, grade_documents_node, generate_answer_node, rewrite_query_node (변수명: RAGState, generate_query_or_respond, grade_documents_node, generate_answer_node, rewrite_query_node)

print("노드 함수 4개 정의 완료")
print("  - generate_query_or_respond (진입)")
print("  - grade_documents_node (관련성 평가)")
print("  - generate_answer_node (답변 생성)")
print("  - rewrite_query_node (쿼리 재작성)")

In [ ]:
# [5-8] : StateGraph 조립 — 노드 등록 및 엣지 연결

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition

# TODO : 함수/클래스 정의: relevance_router (변수명: relevance_router)

print("Agentic RAG StateGraph 컴파일 완료")

In [ ]:
# [5-9] : Agentic RAG StateGraph 실행 테스트

# TODO : Agentic RAG StateGraph 실행 테스트 (변수명: test_q, final_answer, final_answer)

print(f"질문: {test_q}\n")
print("=== 그래프 실행 흐름 ===")
print(f"\n=== 최종 답변 ===")
print(final_answer[:500])

## 6) Re-ranking으로 검색 품질 강화

# [1-6] : Re-ranking으로 검색 품질 강화

**Re-ranking**은 1차 검색된 문서를 LLM이 질문과의 관련성을 기준으로 **재정렬**하는 기법이다. Agent 판단 흐름 내에서 검색 정밀도를 높이는 데 활용된다.

| 단계 | 방식 | 장점 | 단점 |
|------|------|------|------|
| **1차 검색** | 벡터 유사도 (top-k) | 빠름, 대규모 후보 추출 | 정밀도 낮음 |
| **Re-ranking** | LLM 기반 점수 매기기 | 높은 정밀도 | 추가 비용/지연 |

> 진행: [▓▓▓▓▓▓▓░░░] 7/11

In [ ]:
# [5-10] : LLM 기반 Re-ranker 구현 + 검색 품질 비교

# TODO : 함수/클래스 정의: RelevanceScore, rerank_documents (변수명: RelevanceScore, rerank_documents)

print(f"=== 1차 검색 (top-5) ===")
for i, doc in enumerate(candidates):
    print(f"  [{i+1}] topic={doc.metadata.get('topic')} | {doc.page_content[:60]}...")
print(f"\n=== Re-ranking 후 (top-3) ===")
for i, item in enumerate(reranked):
    doc = item["doc"]
    print(f"  [{i+1}] score={item['score']}/10 | topic={doc.metadata.get('topic')}")
    print(f"      이유: {item['reasoning'][:60]}...")

---
# Part 2: Context Memory

Agent의 자율 판단 능력에 더해, **대화 맥락을 유지**하는 Context Memory를 학습한다. Context Engineering의 체계적 프레임워크와 메모리 3유형을 이해하고, `InMemoryStore`로 구현한다.


## 7) Context Engineering 개요

# [2-1] : Context Engineering 개요

**Context Engineering**은 "올바른 정보를, 올바른 형식으로, 올바른 시점에" AI에 제공하는 시스템 설계이다. 단순한 프롬프트 엔지니어링을 넘어, 컨텍스트를 **런타임에 동적으로** 구성하는 아키텍처 접근법이다.

### 2차원 매트릭스 (가변성 x 수명)

| | **단기 (Within Session)** | **장기 (Cross Session)** |
|---|---|---|
| **정적 (Static)** | 시스템 프롬프트, 도구 정의 | 사용자 프로필, 글로벌 규칙 |
| **동적 (Dynamic)** | 대화 히스토리, 검색 결과 | 메모리(Semantic/Episodic/Procedural) |

- **단기 정적**: 한 세션 내에서 변하지 않는 정보 (시스템 프롬프트, 도구 스키마)
- **단기 동적**: 대화가 진행되며 변하는 정보 (메시지 히스토리, RAG 검색 결과)
- **장기 정적**: 세션을 초월하여 유지되는 고정 정보 (사용자 프로필, 도메인 규칙)
- **장기 동적**: 세션을 초월하며 계속 업데이트되는 정보 (학습된 선호도, 과거 경험)

> 진행: [▓▓▓▓▓▓▓▓░░] 8/11

## 8) 메모리 3유형

# [2-2] : 메모리 3유형

인지과학에서 영감을 받은 세 가지 메모리 유형으로 Agent의 장기 기억을 분류한다. 각 유형에 따라 **저장 구조**와 **활용 방식**이 다르다.

| 메모리 유형 | 역할 | 내용 예시 | Namespace 패턴 |
|------------|------|----------|---------------|
| **Semantic** | 사실/프로필 | 사용자 역할, 선호도, 도메인 지식 | `(user_id, "profile")` |
| **Episodic** | 과거 경험 | 이전 대화 요약, 문제 해결 경험 (few-shot) | `(user_id, "episodes")` |
| **Procedural** | 행동 규칙 | 응답 포맷, 도메인 가이드라인 | `(user_id, "procedures")` |

### 활용 패턴
- **Semantic**: 항상 로드 — 사용자가 누구인지 파악
- **Episodic**: 시맨틱 검색으로 관련 경험만 선별 — few-shot 프롬프트에 활용
- **Procedural**: 항상 로드 — 응답 규칙과 가이드라인 적용

> 진행: [▓▓▓▓▓▓▓▓▓░] 9/11

In [ ]:
# [5-11] : InMemoryStore 기본 API — put / get / search

# [2-3] : InMemoryStore + Namespace 패턴

from langgraph.store.memory import InMemoryStore

# 기본 스토어 (키-값 검색)
# TODO : InMemoryStore + Namespace 패턴 (변수명: basic_store, user_id, style_item, all_prefs)

print(f"응답 스타일: {style_item.value}")
print(f"\n저장된 선호도 {len(all_prefs)}개:")
for item in all_prefs:
    print(f"  [{item.key}] = {item.value}")

In [ ]:
# [5-12] : 시맨틱 검색 InMemoryStore — 임베딩 기반 메모리 검색

# [2-4] : 시맨틱 검색 기반 메모리 조회

# 시맨틱 검색 스토어 (임베딩 기반)
# TODO : 시맨틱 검색 기반 메모리 조회 (변수명: semantic_store, index, ns, query1, results1)

print(f"메모리 5개 저장 완료 (namespace={ns})")
print(f"\n쿼리: '{query1}'")
for r in results1:
    print(f"  [{r.key}] {r.value['content']}")
print(f"\n쿼리: '{query2}'")
for r in results2:
    print(f"  [{r.key}] {r.value['content']}")

In [ ]:
# [5-13] : 메모리 3유형 통합 구축 — Semantic / Episodic / Procedural

# TODO : 메모리 3유형 통합 구축 — Semantic / Episodic / Procedural (변수명: mem_store, uid, ep_results)

print("메모리 3유형 저장 완료")
print(f"  Semantic: (uid, 'profile') → 사용자 프로필")
print(f"  Episodic: (uid, 'episodes') → 과거 대화 경험 2개")
print(f"  Procedural: (uid, 'procedures') → 응답 규칙")
print(f"\n관련 과거 경험: {ep_results[0].value['content'] if ep_results else '없음'}")

## 9) Agentic RAG + Context Memory 통합

# [2-5] : Agentic RAG + Context Memory 통합

앞서 구현한 **Agentic RAG StateGraph**와 **Context Memory**를 통합한다. 매 대화마다 메모리에서 관련 맥락을 검색하여 시스템 프롬프트에 주입하고, 대화 경험을 에피소드 메모리에 저장한다.

| 단계 | 설명 |
|------|------|
| 1. 메모리 검색 | 프로필(항상) + 에피소드(시맨틱 검색) + 규칙(항상) |
| 2. 문서 검색 | 벡터스토어에서 관련 문서 top-k 검색 |
| 3. 컨텍스트 결합 | 메모리 + 문서를 시스템 프롬프트에 주입 |
| 4. 답변 생성 | LLM이 풍부한 맥락으로 답변 |
| 5. 경험 저장 | 대화 내용을 에피소드 메모리에 기록 |

> 진행: [▓▓▓▓▓▓▓▓▓▓] 10/11

In [ ]:
# [5-14] : Context Memory + Agentic RAG 통합 — 멀티턴 대화

from langchain_core.messages import SystemMessage
import time

# TODO : 함수/클래스 정의: build_memory_context, memory_rag_chat (변수명: build_memory_context, memory_rag_chat)

for turn, q in enumerate(conversations, 1):
    print(f"\n=== 대화 {turn}: {q} ===")
    answer = memory_rag_chat(mem_store, uid, q)
    print(answer[:350])

---

# [정리] : 핵심 요약

| 항목 | 내용 |
|------|------|
| **다룬 기술** | Agentic RAG, GradeDocuments, Re-ranking, Context Memory, InMemoryStore |
| **핵심 개념** | Agent가 검색 전략을 자율 결정 + 대화 맥락을 메모리로 유지 |
| **GradeDocuments** | `with_structured_output`으로 관련성을 yes/no로 분류 — 조건 분기 기준 |
| **StateGraph 재검색 루프** | `rewrite_query → gen_query` 순환 엣지 — 자기 수정 루프 |
| **Re-ranking** | LLM 기반 점수 매기기로 검색 정밀도 강화 |
| **Context Engineering** | 2차원 매트릭스 (가변성 x 수명) — 올바른 정보를 올바른 시점에 제공 |
| **메모리 3유형** | Semantic(사실) / Episodic(경험) / Procedural(규칙) |
| **InMemoryStore** | `(namespace, key)` 패턴 + 시맨틱 검색 — 사용자별 메모리 격리 |

**Part 1 핵심 — Agentic RAG**
- Naive RAG의 한계 (관련성 검증 부재, 재검색 불가)를 `GradeDocuments` + `StateGraph` 재검색 루프로 극복
- `add_conditional_edges`로 관련성 평가 결과에 따라 동적 라우팅

**Part 2 핵심 — Context Memory**
- Context Engineering 2차원 매트릭스로 컨텍스트의 가변성과 수명을 체계적으로 분류
- `InMemoryStore` + namespace 패턴으로 Semantic/Episodic/Procedural 메모리를 구현
- 시맨틱 검색으로 현재 대화와 관련된 과거 기억을 동적으로 검색

→ 이를 발전시켜 **06_멀티에이전트** — 여러 Agent가 협력하는 아키텍처로 확장